# Issue #84: `IntensityFree` needs *log-space* inter-event time statistics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ant-research/EasyTemporalPointProcess/blob/main/notebooks/easytpp_intensityfree_log_stats.ipynb)

This notebook explains and reproduces the bug reported in
[issue #84](https://github.com/ant-research/EasyTemporalPointProcess/issues/84):
`TPPDataset.get_dt_stats` computed statistics on **raw** inter-event times $\tau$,
but the values are consumed by the `IntensityFree` model as the mean and std of
$\log\tau$. This silently broke the standardization of the log-normal mixture
distribution from the original IFL-TPP paper (Shchur et al., *Intensity-Free
Learning of Temporal Point Processes*, ICLR 2020,
[paper](https://openreview.net/pdf?id=HygOjhEYDH) /
[reference code](https://github.com/shchur/ifl-tpp)).

Along the way we found a **second, independent bug** in the same function: its
streaming mean/variance aggregation updates the sample count *before* using it,
so even the raw-space statistics it returned were wrong.

The notebook is self-contained (only `numpy` and `torch`) so it runs on any
released version of `easy-tpp`. The fix ships in `easy-tpp > 0.2.2`.

**Contents**
1. How the log-normal mixture in IFL-TPP uses these statistics
2. Reproducing bug 1: raw-space instead of log-space statistics
3. Reproducing bug 2: broken streaming aggregation
4. Why it matters: the `log_scales` clamp makes the model unable to compensate
5. The fix, and a workaround for released versions

## 1. How IFL-TPP models inter-event times

`IntensityFree` models the distribution of the next inter-event time $\tau$ with a
**log-normal mixture**. The generative chain (see `dpp/models/log_norm_mix.py` in the
reference repo, mirrored by `LogNormalMixtureDistribution` in EasyTPP) is:

$$x \sim \mathrm{GMM}(\text{locs},\ \text{scales},\ \text{weights})$$
$$y = s \cdot x + m \qquad\leftarrow\ \texttt{AffineTransform}$$
$$\tau = e^{y} \qquad\leftarrow\ \texttt{ExpTransform}$$

The affine parameters $(m, s)$ are **not learned** — they are dataset statistics,
and they are meant to be the mean and standard deviation of $\log\tau$ over the
training set:

```python
# shchur/ifl-tpp, dpp/data/dataset.py
def get_inter_time_statistics(self):
    """Get the mean and std of log(inter_time)."""
    all_inter_times = torch.cat([seq.inter_times[:-1] for seq in self.sequences])
    mean_log_inter_time = all_inter_times.log().mean()
    std_log_inter_time = all_inter_times.log().std()
    return mean_log_inter_time, std_log_inter_time
```

With these values, $y = \log\tau$ is approximately **standardized**: the neural
network only has to produce a GMM close to $\mathcal{N}(0, 1)$, which is exactly
where its initialization puts it. If $(m, s)$ come from the *raw* scale instead,
the base GMM must move to a far-away, badly-scaled region of parameter space —
and, as we show in section 4, a hard clamp in the code can make that literally
impossible.

In [ ]:
import numpy as np
import torch
import torch.distributions as D

rng = np.random.default_rng(0)

# Synthetic dataset: 100 sequences of heavy-tailed inter-event times.
# Ground truth: log(tau) ~ Normal(mu_log, sigma_log), i.e. tau is log-normal.
MU_LOG, SIGMA_LOG = 4.0, 1.5
time_delta_seqs = [rng.lognormal(MU_LOG, SIGMA_LOG, size=rng.integers(40, 120)).tolist()
                   for _ in range(100)]
type_seqs = [[0] * len(s) for s in time_delta_seqs]

all_tau = np.concatenate([np.array(s[1:]) for s in time_delta_seqs])
print(f'true raw-space stats:  mean={all_tau.mean():10.2f}  std={all_tau.std():10.2f}')
print(f'true log-space stats:  mean={np.log(all_tau).mean():10.2f}  std={np.log(all_tau).std():10.2f}')

## 2. Bug 1: statistics computed on the raw scale

Below is the pre-fix `get_dt_stats` copied verbatim from
`easy_tpp/preprocess/dataset.py` (as released up to `easy-tpp 0.2.2`).
Note that `y_bar` and `s_2_y` are computed on `dts` directly — `np.log` never
appears — yet the caller in `easy_tpp/runner/base_runner.py` stores the results
as `mean_log_inter_time` / `std_log_inter_time`:

```python
# easy_tpp/runner/base_runner.py
mean_log_inter_time, std_log_inter_time, min_dt, max_dt = (
    self._data_loader.train_loader().dataset.get_dt_stats())
runner_config.model_config.set("mean_log_inter_time", mean_log_inter_time)
runner_config.model_config.set("std_log_inter_time", std_log_inter_time)
```

In [ ]:
def get_dt_stats_old(time_delta_seqs, type_seqs):
    """Verbatim copy of TPPDataset.get_dt_stats from easy-tpp <= 0.2.2 (prints removed)."""
    x_bar, s_2_x, n = 0., 0., 0
    min_dt, max_dt = np.inf, -np.inf

    for dts, marks in zip(time_delta_seqs, type_seqs):
        dts = np.array(dts[1:-1 if marks[-1] == -1 else None])
        min_dt = min(min_dt, dts.min())
        max_dt = max(max_dt, dts.max())
        y_bar = dts.mean()          # <-- mean of RAW tau, not log(tau)
        s_2_y = dts.var()           # <-- var  of RAW tau, not log(tau)
        m = dts.shape[0]
        n += m                      # <-- bug 2: incremented BEFORE the formulas below use it
        s_2_x = (((n - 1) * s_2_x + (m - 1) * s_2_y) / (n + m - 1)) + (
                    (n * m * ((x_bar - y_bar) ** 2)) / ((n + m) * (n + m - 1)))
        x_bar = (n * x_bar + m * y_bar) / (n + m)

    return x_bar, (s_2_x ** 0.5), min_dt, max_dt


mean_old, std_old, _, _ = get_dt_stats_old(time_delta_seqs, type_seqs)
print(f'old get_dt_stats returns:      mean={mean_old:10.2f}  std={std_old:10.2f}')
print(f'model expects (log-space):     mean={np.log(all_tau).mean():10.2f}  std={np.log(all_tau).std():10.2f}')

The values handed to the model are off by roughly the ratio between the raw and
log scales — two orders of magnitude here, and arbitrarily more on datasets with
heavier tails or larger time units (seconds vs. days).

## 3. Bug 2: the streaming aggregation itself is wrong

The chunked mean/variance combination (Chan et al. style) requires `n` to be the
count *before* merging the new chunk, but the code runs `n += m` first. The
cleanest way to see it: feed a **single** sequence. The correct answer is simply
that sequence's mean — the old code returns **half** of it:

with `x_bar = 0` and `n` already incremented to `m`,
$$\bar{x} \leftarrow \frac{n\,\bar{x} + m\,\bar{y}}{n + m} = \frac{m \cdot 0 + m\,\bar{y}}{2m} = \frac{\bar{y}}{2}$$

In [ ]:
one_seq = [time_delta_seqs[0]]
one_marks = [type_seqs[0]]
true_mean = np.array(one_seq[0][1:]).mean()
mean_one, std_one, _, _ = get_dt_stats_old(one_seq, one_marks)
print(f'single-sequence dataset: true mean = {true_mean:8.2f}')
print(f'old get_dt_stats mean   = {mean_one:8.2f}   (exactly half: {true_mean / 2:8.2f})')

## 4. Why it matters: the model cannot compensate

One might hope the network absorbs a wrong-but-invertible affine transform by
learning shifted `locs` and shrunken `scales`. Two things prevent that in
practice:

1. **Initialization / optimization**: at init the GMM sits near
   $\mathcal{N}(0,1)$; with raw stats the data lives at
   $x = (\log\tau - m_{raw})/s_{raw}$, a thin sliver far from where gradients
   start.
2. **A hard clamp**: `IntensityFree` clamps `log_scales` to `[-5, 3]`
   (`clamp_preserve_gradients(log_scales, -5.0, 3.0)`). After the affine
   transform, the smallest standard deviation the model can express in log-space
   is $s_{raw} \cdot e^{-5}$. When that floor exceeds the true
   $\sigma_{\log\tau}$, **no parameter setting can fit the data** — the
   likelihood stays catastrophically low, exactly the symptom reported in the
   issue comments.

Let's measure both effects with the actual distribution object (rebuilt with
plain `torch.distributions`; identical math to EasyTPP's
`LogNormalMixtureDistribution.log_prob`).

In [ ]:
def log_normal_mixture(locs, log_scales, log_weights, mean, std):
    """Same chain as easy_tpp LogNormalMixtureDistribution: GMM -> Affine -> Exp."""
    gmm = D.MixtureSameFamily(D.Categorical(logits=log_weights),
                              D.Normal(loc=locs, scale=log_scales.exp()))
    return D.TransformedDistribution(
        gmm, [D.AffineTransform(loc=mean, scale=std), D.ExpTransform()])

tau = torch.tensor(all_tau, dtype=torch.float64).clamp(min=1e-5)

# A freshly initialized (or lightly trained) network outputs a near-standard base GMM.
locs = torch.zeros(1, dtype=torch.float64)
log_scales = torch.zeros(1, dtype=torch.float64)
log_weights = torch.zeros(1, dtype=torch.float64)

mean_log, std_log = np.log(all_tau).mean(), np.log(all_tau).std()

nll_correct = -log_normal_mixture(locs, log_scales, log_weights, mean_log, std_log).log_prob(tau).mean()
nll_buggy   = -log_normal_mixture(locs, log_scales, log_weights, mean_old, std_old).log_prob(tau).mean()

print(f'mean NLL per event with LOG-space stats (correct): {nll_correct.item():12.2f}')
print(f'mean NLL per event with RAW-space stats (buggy):   {nll_buggy.item():12.2f}')

In [ ]:
# Best case AFTER training: the network fully compensates the wrong affine, i.e.
# locs = (log tau stats - mean_raw)/std_raw, scale = sigma_log/std_raw ... unless the clamp bites.
print('required log_scale to compensate vs. the [-5, 3] clamp:\n')
print(f'{"raw std of dataset":>20} {"required log_scale":>20} {"min std floor (raw stats)":>28} {"fits?":>7}')
for s_raw in [std_old, 1e3, 1e4, 1e5]:
    required = np.log(SIGMA_LOG / s_raw)   # log_scale the net must output
    floor = s_raw * np.exp(-5.0)           # smallest achievable std of log(tau)
    print(f'{s_raw:20.1f} {required:20.2f} {floor:28.3f} {str(required >= -5.0):>7}')
print(f'\n(true sigma of log(tau) is {SIGMA_LOG}; once the floor exceeds it, no parameters can fit the data)')

The initialized model starts ~5 nats/event worse — and it can never close the
gap: even on this moderate dataset the `log_scale` needed to compensate
(−5.51) already violates the clamp, so the smallest expressible log-space std
(≈2.5) exceeds the true $\sigma_{\log\tau} = 1.5$. The correct model keeps
improving with training while the buggy one plateaus far above it, and the
deficit grows with the raw scale of the dataset (std in the thousands is common
with timestamps in seconds). This matches the empirical report in the issue —
very large negative log-likelihood, fixed by monkey-patching log-space
statistics.

## 5. The fix

Fixed in EasyTPP after `0.2.2` ([issue #84](https://github.com/ant-research/EasyTemporalPointProcess/issues/84)).
Since `time_delta_seqs` is already fully in memory, the streaming formula was
dropped entirely:

```python
def get_dt_stats(self):
    all_dts = np.concatenate([
        np.array(dts[1:-1 if marks[-1] == -1 else None])
        for dts, marks in zip(self.time_delta_seqs, self.type_seqs)
    ])
    min_dt, max_dt = all_dts.min(), all_dts.max()
    log_dts = np.log(np.clip(all_dts, 1e-5, None))  # matches inter_times.clamp(min=1e-5) in IntensityFree.loglike_loss
    return log_dts.mean(), log_dts.std(), min_dt, max_dt
```

The `1e-5` clip mirrors `inter_times.clamp(min=1e-5)` inside
`IntensityFree.loglike_loss`, so the statistics describe exactly the quantities
the likelihood sees (and `log(0)` from simultaneous events is avoided).

In [ ]:
def get_dt_stats_fixed(time_delta_seqs, type_seqs):
    all_dts = np.concatenate([
        np.array(dts[1:-1 if marks[-1] == -1 else None])
        for dts, marks in zip(time_delta_seqs, type_seqs)
    ])
    min_dt, max_dt = all_dts.min(), all_dts.max()
    log_dts = np.log(np.clip(all_dts, 1e-5, None))
    return log_dts.mean(), log_dts.std(), min_dt, max_dt


mean_fix, std_fix, _, _ = get_dt_stats_fixed(time_delta_seqs, type_seqs)
print(f'fixed get_dt_stats:  mean_log={mean_fix:.4f}  std_log={std_fix:.4f}')
print(f'ground truth:        mean_log={np.log(all_tau).mean():.4f}  std_log={np.log(all_tau).std():.4f}')
print(f'(generating parameters were mu={MU_LOG}, sigma={SIGMA_LOG})')

### Workaround on released versions (`easy-tpp <= 0.2.2`)

Until you upgrade, set the statistics on the model before training, as in the
issue comments:

```python
model_runner.model.mean_log_inter_time = mean_log_inter_time   # from get_dt_stats_fixed
model_runner.model.std_log_inter_time = std_log_inter_time
```

**Note on checkpoints:** `IntensityFree` models trained before the fix bake the
old statistics into the distribution, so their metrics are not comparable with
models trained after it — retrain to benefit from the fix.

### References
- O. Shchur, M. Biloš, S. Günnemann. *Intensity-Free Learning of Temporal Point Processes.* ICLR 2020. [openreview](https://openreview.net/pdf?id=HygOjhEYDH)
- Reference implementation: [shchur/ifl-tpp](https://github.com/shchur/ifl-tpp)
- [EasyTPP issue #84](https://github.com/ant-research/EasyTemporalPointProcess/issues/84)